$$
PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{\,2i/d_{\text{model}}}}\right)\\
PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{\,2i/d_{\text{model}}}}\right)\\
H^{(0)} = X + PE \in \mathbb{R}^{n \times d_{\text{model}}}
$$

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V\\
S = QK^{\top} \in \mathbb{R}^{n \times n}\\
S_{\text{scaled}} = \frac{S}{\sqrt{d_k}}\\
A = \text{softmax}(S_{\text{scaled}}) \in \mathbb{R}^{n \times n}\\
O = AV \in \mathbb{R}^{n \times d_v}\\
\text{Var}\left(\frac{q \cdot k}{\sqrt{d_k}}\right) = \left(\frac{1}{\sqrt{d_k}}\right)^2 \cdot d_k = 1
$$

$$
Q_i = HW_i^Q, \quad K_i = HW_i^K, \quad V_i = HW_i^V\\
W_i^Q \in \mathbb{R}^{d_{\text{model}} \times d_k}\\
W_i^K \in \mathbb{R}^{d_{\text{model}} \times d_k}\\
W_i^V \in \mathbb{R}^{d_{\text{model}} \times d_v}\\
\text{head}_i = \text{Attention}(Q_i,\, K_i,\, V_i) = \text{softmax}\!\left(\frac{Q_i K_i^\top}{\sqrt{d_k}}\right) V_i\\
\text{Concat}(\text{head}_1, \ldots, \text{head}_h) \in \mathbb{R}^{n \times (h \cdot d_v)}\\
\text{MultiHead}(H) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\, W^O\\
$$

$$\mathrm{FFN}(x) = \max\!\bigl(0,\; x\,W_1 + b_1\bigr)\,W_2 + b_2\\
\mathrm{GELU}(x) = x \cdot \Phi(x) = x \cdot \frac{1}{2}\left[1 + \mathrm{erf}\!\left(\frac{x}{\sqrt{2}}\right)\right]\\
\mathrm{erf}(x) = \frac{2}{\sqrt{\pi}} \int_{0}^{x} e^{-t^2}\, dt\\
\Phi(x) = P(Z \leq x), \quad Z \sim \mathcal{N}(0,\,1)\\
\lim_{x \to +\infty} \Phi(x) = 1 \;\Longrightarrow\; \mathrm{GELU}(x) = x\cdot\Phi(x) \approx x \quad \text{（近似线性）}\\
\lim_{x \to -\infty} \Phi(x) = 0 \;\Longrightarrow\; \mathrm{GELU}(x) = x\cdot\Phi(x) \approx 0 \quad \text{（被抑制）}\\
\mathrm{Swish}(x) = x \cdot \sigma(x) = x \cdot \frac{1}{1 + e^{-x}}\\
\mathrm{SwiGLU}(x) = \underbrace{(xW_1 + b_1)}_{\text{信息流}} \;\otimes\; \underbrace{\mathrm{Swish}(xW_2 + b_2)}_{\text{门控}}\\
\mathrm{FFN}_{\mathrm{SwiGLU}}(x) = \left(\mathrm{Swish}(xW_1) \;\otimes\; xW_2\right) W_3
$$

$$
x_{\text{out}} = x + \mathrm{Sublayer}(x)\\
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial x_{\text{out}}} \cdot \frac{\partial x_{\text{out}}}{\partial x} = \frac{\partial L}{\partial x_{\text{out}}} \cdot \left(I + \frac{\partial\,\mathrm{Sublayer}(x)}{\partial x}\right)$$

$$\mathrm{LayerNorm}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta\\
\mu = \frac{1}{d}\sum_{i=1}^{d} x_i, \qquad \sigma^2 = \frac{1}{d}\sum_{i=1}^{d} (x_i - \mu)^2
$$

$$\mathrm{CrossAttn}(H_d, H_e) = \mathrm{MultiHead}(Q, K, V)\\
Q = H_d W_Q, \quad K = H_e W_K, \quad V = H_e W_V\\
H_d \in \mathbb{R}^{n_t \times d} \;\text{：解码器隐藏状态（提供 Query）}\\
H_e \in \mathbb{R}^{n_s \times d} \;\text{：编码器输出（提供 Key 和 Value）}
$$

$$M \in \{0, 1\}^{n \times n}\\
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d}} + M\right) V\\
M_{ij} = \begin{cases} 
, & \text{if } j \leq i \quad \text{（可见）} \\ 
-\infty, & \text{if } j > i \quad \text{（遮蔽）} 
\end{cases}
$$

In [4]:
#配置与词表
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
D = 512
H = 8
DK = D // H
FF = 2048
N = 6
ZH = {'<pad>':0,'<bos>':1,'<eos>':2,'我':3,'爱':4,'机器':5,'学习':6,'你':7,'好':8}
EH = {'<pad>':0,'<bos>':1,'<eos>':2,'I':3,'love':4,'machine':5,'learning':6,'you':7,'hello':8}
V_ZH,V_EH = len(ZH),len(EH)
PAD,BOS,EOS = 0,1,2
#分词
def tokenize_zh(text):
    return [ZH[w] for w in text.split()]
def detokenize_en(ids):
    inv = {v:k for k,v in EH.items()}
    return ' '.join(inv[i] for i in ids if i > EOS)
ids = tokenize_zh('我 爱 机器 学习')
print(ids)
#位置编码
def positional_encoding(max_s,d=D):
    pe = torch.zeros(max_s,d)
    pos = torch.arange(max_s).float().unsqueeze(1)
    div = torch.exp(torch.arange(0,d,2).float() * (-math.log(10000.0)/d))
    pe[:,0::2] = torch.sin(pos * div)
    pe[:,1::2] = torch.cos(pos * div)

[3, 4, 5, 6]
